# Session 8.1: Text Preprocessing Pipeline

## Learning Objectives
1. Understand text as data
2. Perform text preprocessing
3. Master tokenization, stemming, lemmatization
4. Build reusable text preprocessing pipeline
5. Understand NLP applications

**Duration:** 2 hours  
**Target:** Build complete text preprocessing pipeline

## 1. Setup and Installations

### LLM Prompt for Installation:
```
Create a Python code block that installs NLTK, spaCy, and wordcloud libraries.
Also download the required NLTK data: punkt, stopwords, wordnet, averaged_perceptron_tagger.
Include error handling for the downloads.
```

In [ ]:
# Install required libraries
!pip install nltk spacy wordcloud matplotlib seaborn pandas -q

# Download spaCy English model
!python -m spacy download en_core_web_sm -q

In [ ]:
# Import libraries
import nltk
import spacy
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# Download NLTK data
nltk_data = ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4']
for data in nltk_data:
    try:
        nltk.download(data, quiet=True)
    except Exception as e:
        print(f"Error downloading {data}: {e}")

# Import NLTK modules
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully!")

## 2. Sample Text Data

Let's create sample texts to work with throughout this notebook.

In [ ]:
# Sample texts from different domains
sample_texts = [
    """Natural Language Processing (NLP) is revolutionizing how computers understand human language! 
    It's amazing how machines can now analyze, interpret, and generate text. #AI #MachineLearning
    Visit https://www.nlp.com for more info @MLEnthusiast""",
    
    """The stock market is experiencing unprecedented volatility. Investors are closely watching 
    tech stocks, which have fallen 15% this quarter. Analysts predict further turbulence ahead.""",
    
    """I absolutely LOVED this movie!!! The acting was phenomenal, the plot was engaging, 
    and the cinematography was stunning. Highly recommend watching it. 10/10!!!""",
    
    """Running, runner, runs, ran - these words all share the same root. 
    Similarly, better, good, best are related forms. The English language has many such variations.""",
    
    """URGENT!!! You've WON $1,000,000!!! Click here NOW to claim your prize!!! 
    This offer expires in 24 hours!!! ACT FAST!!!"""
]

print("Sample Texts Loaded:")
for i, text in enumerate(sample_texts, 1):
    print(f"\nText {i}:\n{text[:100]}...")

## 3. Tokenization

Breaking text into individual words (tokens) or sentences.

### LLM Prompt:
```
Create functions to demonstrate word tokenization and sentence tokenization using NLTK.
Show the difference between simple split() and proper tokenization.
```

In [ ]:
# Demonstrate word tokenization
text = sample_texts[0]

print("Original Text:")
print(text)
print("\n" + "="*80)

# Simple split vs proper tokenization
print("\n1. Simple .split() method:")
simple_tokens = text.split()
print(f"Tokens: {simple_tokens[:10]}")
print(f"Count: {len(simple_tokens)}")

print("\n2. NLTK word_tokenize():")
nltk_tokens = word_tokenize(text)
print(f"Tokens: {nltk_tokens[:15]}")
print(f"Count: {len(nltk_tokens)}")

print("\n3. spaCy tokenization:")
doc = nlp(text)
spacy_tokens = [token.text for token in doc]
print(f"Tokens: {spacy_tokens[:15]}")
print(f"Count: {len(spacy_tokens)}")

print("\n" + "="*80)
print("Notice: NLTK and spaCy handle punctuation separately!")

In [ ]:
# Sentence tokenization
long_text = sample_texts[1]

print("Original Text:")
print(long_text)
print("\n" + "="*80)

sentences = sent_tokenize(long_text)
print(f"\nNumber of sentences: {len(sentences)}")
print("\nSentences:")
for i, sent in enumerate(sentences, 1):
    print(f"{i}. {sent}")

## 4. Text Cleaning

Removing unwanted elements from text.

In [ ]:
def clean_text(text):
    """
    Comprehensive text cleaning function.
    
    Args:
        text (str): Raw text to clean
    
    Returns:
        str: Cleaned text
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)
    
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove numbers (optional - depends on use case)
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Demonstrate cleaning on all sample texts
print("BEFORE AND AFTER CLEANING:\n")
for i, text in enumerate(sample_texts, 1):
    print(f"Text {i}:")
    print(f"BEFORE: {text[:150]}...")
    cleaned = clean_text(text)
    print(f"AFTER:  {cleaned[:150]}...")
    print("\n" + "-"*80 + "\n")

## 5. Stop Words Removal

Removing common words that don't carry much meaning ("the", "is", "at", etc.)

In [ ]:
# Load stop words
stop_words = set(stopwords.words('english'))

print(f"Total stop words: {len(stop_words)}")
print(f"\nSample stop words: {list(stop_words)[:30]}")

# Function to remove stop words
def remove_stopwords(text):
    """
    Remove stop words from text.
    
    Args:
        text (str): Input text
    
    Returns:
        str: Text with stop words removed
    """
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
    return ' '.join(filtered_tokens)

# Demonstrate stop word removal
print("\n" + "="*80)
print("STOP WORD REMOVAL DEMONSTRATION:\n")

text = sample_texts[1]
cleaned = clean_text(text)

print("BEFORE (cleaned):")
print(cleaned)
print(f"\nWord count: {len(cleaned.split())}")

without_stopwords = remove_stopwords(cleaned)
print("\nAFTER (stop words removed):")
print(without_stopwords)
print(f"\nWord count: {len(without_stopwords.split())}")
print(f"\nReduction: {len(cleaned.split()) - len(without_stopwords.split())} words removed")

## 6. Stemming vs Lemmatization

**Stemming:** Crude chopping of word endings (running → run)  
**Lemmatization:** Proper reduction to base form using dictionary (better → good)

In [ ]:
# Initialize stemmer and lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Test words
test_words = [
    'running', 'runs', 'ran', 'runner',
    'better', 'good', 'best',
    'caring', 'cares', 'cared',
    'flies', 'flying', 'flew',
    'studies', 'studying', 'studied'
]

print("STEMMING vs LEMMATIZATION COMPARISON:\n")
print(f"{'Original':<15} {'Stemmed':<15} {'Lemmatized':<15}")
print("-" * 45)

for word in test_words:
    stemmed = stemmer.stem(word)
    lemmatized = lemmatizer.lemmatize(word, pos='v')  # v = verb
    print(f"{word:<15} {stemmed:<15} {lemmatized:<15}")

print("\nKey Observations:")
print("- Stemming is faster but rougher (e.g., 'better' → 'better')")
print("- Lemmatization is slower but more accurate (e.g., 'better' → 'good')")
print("- Stemming may produce non-words (e.g., 'studi')")
print("- Lemmatization always produces valid words")

In [ ]:
# Apply to full text
text = "The runner is running faster. He runs better than his competitors and ran the best race."

print("Original Text:")
print(text)

# Tokenize
tokens = word_tokenize(text.lower())
tokens = [t for t in tokens if t.isalpha()]

# Stemming
stemmed = [stemmer.stem(token) for token in tokens]
print("\nStemmed:")
print(' '.join(stemmed))

# Lemmatization
lemmatized = [lemmatizer.lemmatize(token, pos='v') for token in tokens]
print("\nLemmatized:")
print(' '.join(lemmatized))

## 7. Complete Preprocessing Pipeline

Combining all steps into a reusable function.

In [ ]:
def preprocess_text(text, 
                   lowercase=True,
                   remove_urls=True,
                   remove_mentions=True,
                   remove_hashtags=True,
                   remove_numbers=True,
                   remove_punctuation=True,
                   remove_stopwords=True,
                   lemmatize=True):
    """
    Complete text preprocessing pipeline.
    
    Args:
        text (str): Raw text to preprocess
        lowercase (bool): Convert to lowercase
        remove_urls (bool): Remove URLs
        remove_mentions (bool): Remove @mentions
        remove_hashtags (bool): Remove #hashtags
        remove_numbers (bool): Remove numbers
        remove_punctuation (bool): Remove punctuation
        remove_stopwords (bool): Remove stop words
        lemmatize (bool): Apply lemmatization
    
    Returns:
        str: Preprocessed text
    """
    # Lowercase
    if lowercase:
        text = text.lower()
    
    # Remove URLs
    if remove_urls:
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove mentions
    if remove_mentions:
        text = re.sub(r'@\w+', '', text)
    
    # Remove hashtags
    if remove_hashtags:
        text = re.sub(r'#\w+', '', text)
    
    # Remove numbers
    if remove_numbers:
        text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    if remove_punctuation:
        text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stop words
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [t for t in tokens if t.lower() not in stop_words and t.isalpha()]
    
    # Lemmatization
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(token, pos='v') for token in tokens]
    
    # Remove extra whitespace and join
    text = ' '.join(tokens)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test the pipeline
print("COMPLETE PREPROCESSING PIPELINE:\n")
for i, text in enumerate(sample_texts, 1):
    print(f"Text {i}:")
    print(f"ORIGINAL: {text[:100]}...")
    processed = preprocess_text(text)
    print(f"PROCESSED: {processed[:100]}...")
    print("\n" + "-"*80 + "\n")

## 8. Visualizations

### 8.1 Word Cloud

In [ ]:
# Combine all texts
combined_text = ' '.join(sample_texts)

# Create word clouds before and after preprocessing
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before preprocessing
wordcloud_before = WordCloud(width=800, height=400, 
                             background_color='white',
                             colormap='viridis').generate(combined_text)

axes[0].imshow(wordcloud_before, interpolation='bilinear')
axes[0].set_title('Word Cloud: Before Preprocessing', fontsize=14, fontweight='bold')
axes[0].axis('off')

# After preprocessing
processed_text = preprocess_text(combined_text)
wordcloud_after = WordCloud(width=800, height=400,
                           background_color='white',
                           colormap='plasma').generate(processed_text)

axes[1].imshow(wordcloud_after, interpolation='bilinear')
axes[1].set_title('Word Cloud: After Preprocessing', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("Notice: After preprocessing, meaningful words stand out more!")

### 8.2 Word Frequency Analysis

In [ ]:
# Word frequency before and after
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Before preprocessing
tokens_before = word_tokenize(combined_text.lower())
tokens_before = [t for t in tokens_before if t.isalpha()]
freq_before = Counter(tokens_before)
top_words_before = freq_before.most_common(15)

words, counts = zip(*top_words_before)
axes[0].barh(range(len(words)), counts, color='skyblue')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].set_xlabel('Frequency', fontsize=11)
axes[0].set_title('Top 15 Words: Before Preprocessing', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()

# After preprocessing
tokens_after = processed_text.split()
freq_after = Counter(tokens_after)
top_words_after = freq_after.most_common(15)

words, counts = zip(*top_words_after)
axes[1].barh(range(len(words)), counts, color='coral')
axes[1].set_yticks(range(len(words)))
axes[1].set_yticklabels(words)
axes[1].set_xlabel('Frequency', fontsize=11)
axes[1].set_title('Top 15 Words: After Preprocessing', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("Observation: Stop words dominate before preprocessing!")

### 8.3 Token Length Distribution

In [ ]:
# Analyze token lengths
token_lengths_before = [len(token) for token in tokens_before]
token_lengths_after = [len(token) for token in tokens_after]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Before
axes[0].hist(token_lengths_before, bins=20, color='lightblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Token Length', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Token Length Distribution: Before Preprocessing', fontsize=13, fontweight='bold')
axes[0].axvline(np.mean(token_lengths_before), color='red', linestyle='--', 
                label=f'Mean: {np.mean(token_lengths_before):.1f}')
axes[0].legend()

# After
axes[1].hist(token_lengths_after, bins=20, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Token Length', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Token Length Distribution: After Preprocessing', fontsize=13, fontweight='bold')
axes[1].axvline(np.mean(token_lengths_after), color='red', linestyle='--',
                label=f'Mean: {np.mean(token_lengths_after):.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### 8.4 Preprocessing Impact Statistics

In [ ]:
# Calculate statistics
stats_data = []

for i, text in enumerate(sample_texts, 1):
    original_len = len(text)
    original_words = len(word_tokenize(text))
    
    processed = preprocess_text(text)
    processed_len = len(processed)
    processed_words = len(processed.split())
    
    stats_data.append({
        'Text': f'Text {i}',
        'Original Chars': original_len,
        'Processed Chars': processed_len,
        'Char Reduction %': round((1 - processed_len/original_len) * 100, 1),
        'Original Words': original_words,
        'Processed Words': processed_words,
        'Word Reduction %': round((1 - processed_words/original_words) * 100, 1)
    })

stats_df = pd.DataFrame(stats_data)
print("PREPROCESSING IMPACT STATISTICS:\n")
print(stats_df.to_string(index=False))

# Visualize reduction
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(stats_df))
width = 0.35

ax.bar(x - width/2, stats_df['Original Words'], width, label='Before', color='skyblue')
ax.bar(x + width/2, stats_df['Processed Words'], width, label='After', color='coral')

ax.set_xlabel('Text Sample', fontsize=11)
ax.set_ylabel('Word Count', fontsize=11)
ax.set_title('Word Count: Before vs After Preprocessing', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(stats_df['Text'])
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 8.5 spaCy Advanced Features

In [ ]:
# Demonstrate spaCy's additional features
text = sample_texts[0]
doc = nlp(text)

print("spaCy ADVANCED FEATURES:\n")
print("Original Text:")
print(text)
print("\n" + "="*80)

# Part-of-speech tagging
print("\n1. Part-of-Speech Tagging:")
print(f"{'Token':<20} {'POS':<10} {'Tag':<10} {'Explanation':<30}")
print("-" * 70)
for token in doc[:15]:
    if token.text.strip():  # Skip whitespace
        print(f"{token.text:<20} {token.pos_:<10} {token.tag_:<10} {spacy.explain(token.tag_) or 'N/A':<30}")

# Named Entity Recognition
print("\n2. Named Entity Recognition:")
if doc.ents:
    print(f"{'Entity':<30} {'Type':<15} {'Explanation':<30}")
    print("-" * 75)
    for ent in doc.ents:
        print(f"{ent.text:<30} {ent.label_:<15} {spacy.explain(ent.label_):<30}")
else:
    print("No named entities found.")

# Lemmatization with spaCy
print("\n3. Lemmatization:")
print(f"{'Token':<20} {'Lemma':<20}")
print("-" * 40)
for token in doc[:15]:
    if token.text.strip() and token.is_alpha:
        print(f"{token.text:<20} {token.lemma_:<20}")

## 9. Real-World Application: Tweet Preprocessing

In [ ]:
# Sample tweets
tweets = [
    "Just got my #iPhone15!!! 📱 Love it!!! @Apple you rock! https://apple.com/iphone",
    "Can't believe I won $500 at the lottery!!! 💰💰💰 Best day ever!!!",
    "@CustomerService your service is TERRIBLE!!! I've been waiting for 2 hours! #frustrated",
    "Beautiful sunset 🌅 at the beach today. So peaceful... #nature #photography",
    "Breaking: Stock market crashes 15% 📉 Investors panic!!! #finance #news"
]

# Preprocess tweets
print("TWEET PREPROCESSING EXAMPLE:\n")
processed_tweets = []

for i, tweet in enumerate(tweets, 1):
    print(f"Tweet {i}:")
    print(f"ORIGINAL: {tweet}")
    
    processed = preprocess_text(tweet)
    processed_tweets.append(processed)
    
    print(f"PROCESSED: {processed}")
    print("\n" + "-"*80 + "\n")

# Create word cloud from tweets
all_tweets = ' '.join(processed_tweets)
wordcloud = WordCloud(width=1200, height=600,
                     background_color='white',
                     colormap='coolwarm').generate(all_tweets)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Word Cloud from Processed Tweets', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 10. Key Takeaways

### What We Learned:
1. **Tokenization**: Breaking text into words and sentences
2. **Text Cleaning**: Removing URLs, mentions, hashtags, punctuation
3. **Stop Words**: Filtering out common words that add little meaning
4. **Stemming vs Lemmatization**: Two approaches to reducing words to their base forms
5. **Complete Pipeline**: Combining all steps into a reusable function

### Best Practices:
- **Always lowercase** for consistency (unless case matters)
- **Keep numbers** if they're meaningful (e.g., product reviews, financial text)
- **Choose stemming** for speed, **lemmatization** for accuracy
- **Customize preprocessing** based on your specific task
- **Visualize** to understand the impact of preprocessing

### When to Use What:
- **Sentiment Analysis**: Remove stop words, lemmatize
- **Topic Modeling**: Keep stop words might be okay, lemmatize
- **Search/IR**: Stemming is often sufficient
- **Translation**: Minimal preprocessing

### Common Pitfalls:
- Over-preprocessing (losing too much information)
- Under-preprocessing (too much noise)
- Not validating preprocessing results
- Using same preprocessing for all tasks

## 11. Practice Exercise

Create your own text preprocessing pipeline for a specific use case!

In [ ]:
# Exercise: Customize the preprocessing pipeline
# Try different combinations of preprocessing steps

custom_text = """
Enter your own text here and experiment with different preprocessing options!
Try tweets, news articles, product reviews, or any text you're interested in.
"""

# Experiment with different parameters
processed = preprocess_text(
    custom_text,
    lowercase=True,
    remove_urls=True,
    remove_mentions=True,
    remove_hashtags=True,
    remove_numbers=False,  # Try changing this
    remove_punctuation=True,
    remove_stopwords=True,
    lemmatize=True
)

print("Your Processed Text:")
print(processed)

## Success Criteria

- ✅ Understand text tokenization (word and sentence level)
- ✅ Clean text effectively (remove URLs, mentions, etc.)
- ✅ Apply stop word removal
- ✅ Understand stemming vs lemmatization tradeoffs
- ✅ Build complete, reusable preprocessing pipeline
- ✅ Visualize preprocessing impact
- ✅ Apply to real-world text (tweets, reviews, etc.)

**Next Session:** Sentiment Analysis with Bag of Words and TF-IDF!